In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"使用设备: {device}")

# 训练集：加入随机翻转和旋转，增加模型见识
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), #随机水平翻转（左右镜像）
    transforms.RandomRotation(10), #随机旋转+—10度
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 测试集：不增强，只做标准化
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

train_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=train_transform)
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

使用设备: mps


In [ ]:
class ProCNN(nn.Module):
    def __init__(self):
        super(ProCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) #归一化

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3) 
        
        # 经过 3 次池化: 28x28 -> 14x14 -> 7x7 -> 3x3 (注意最后尺寸是 3x3)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        

        x = x.view(-1, 128 * 3 * 3)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = ProCNN().to(device)

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad() #清空上一轮的梯度
        pred = model(X)
        loss = loss_fn(pred, y) 
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def test_epoch(model, loader, loss_fn, device):
    model.eval()
    correct = 0
    with torch.no_grad(): #禁用梯度计算
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    return correct / len(loader.dataset)

In [ ]:
epochs = 20 
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 学习率调度器：每 10 轮将学习率乘以 0.1
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print("开始训练...")
for t in range(epochs):
    avg_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    acc = test_epoch(model, test_loader, loss_fn, device)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {t+1:2d} | Loss: {avg_loss:.4f} | Acc: {acc:>0.2%} | LR: {current_lr:.6f}")
    scheduler.step()

print("训练全部完成！")

每7轮，lr*0.2<br>
开始训练...<br>
Epoch  1 | Loss: 0.4906 | Acc: 87.10% | LR: 0.001000<br>
Epoch  2 | Loss: 0.3380 | Acc: 88.73% | LR: 0.001000<br>
Epoch  3 | Loss: 0.3051 | Acc: 89.18% | LR: 0.001000<br>
Epoch  4 | Loss: 0.2752 | Acc: 90.11% | LR: 0.001000<br>
Epoch  5 | Loss: 0.2617 | Acc: 90.69% | LR: 0.001000<br>
Epoch  6 | Loss: 0.2533 | Acc: 91.14% | LR: 0.001000<br>
Epoch  7 | Loss: 0.2420 | Acc: 91.02% | LR: 0.001000<br>
Epoch  8 | Loss: 0.2074 | Acc: 92.11% | LR: 0.000200<br>
Epoch  9 | Loss: 0.1982 | Acc: 92.41% | LR: 0.000200<br>
Epoch 10 | Loss: 0.1941 | Acc: 92.27% | LR: 0.000200<br>
Epoch 11 | Loss: 0.1900 | Acc: 92.35% | LR: 0.000200<br>
Epoch 12 | Loss: 0.1880 | Acc: 92.33% | LR: 0.000200<br>
Epoch 13 | Loss: 0.1843 | Acc: 92.51% | LR: 0.000200<br>
Epoch 14 | Loss: 0.1824 | Acc: 92.79% | LR: 0.000200<br>
Epoch 15 | Loss: 0.1730 | Acc: 92.87% | LR: 0.000040<br>
Epoch 16 | Loss: 0.1693 | Acc: 92.91% | LR: 0.000040<br>
Epoch 17 | Loss: 0.1697 | Acc: 92.86% | LR: 0.000040<br>
Epoch 18 | Loss: 0.1687 | Acc: 92.90% | LR: 0.000040<br>
Epoch 19 | Loss: 0.1669 | Acc: 92.94% | LR: 0.000040<br>
Epoch 20 | Loss: 0.1669 | Acc: 92.98% | LR: 0.000040<br>
训练全部完成！<br>

数据增强的“副作用”：它的 泛化能力变强了，但收敛速度变慢了。<br>
20 轮可能还不够：通常会跑 50 甚至 100 轮。因为模型每一轮都在见“新题”，它需要更多时间去消化。<br>
学习率掉得太快：scheduler 让学习率在第 14 轮变得太小，模型后面就“跑不动”了，没力气去冲刺 94%。<br>

每10轮，lr*0.1<br>
开始训练...<br>
Epoch  1 | Loss: 0.2152 | Acc: 91.13% | LR: 0.001000<br>
Epoch  2 | Loss: 0.2139 | Acc: 91.95% | LR: 0.001000<br>
Epoch  3 | Loss: 0.2087 | Acc: 92.01% | LR: 0.001000<br>
Epoch  4 | Loss: 0.2039 | Acc: 91.60% | LR: 0.001000<br>
Epoch  5 | Loss: 0.1981 | Acc: 92.56% | LR: 0.001000<br>
Epoch  6 | Loss: 0.1922 | Acc: 92.89% | LR: 0.001000<br>
Epoch  7 | Loss: 0.1902 | Acc: 92.12% | LR: 0.001000<br>
Epoch  8 | Loss: 0.1841 | Acc: 92.56% | LR: 0.001000<br>
Epoch  9 | Loss: 0.1823 | Acc: 92.74% | LR: 0.001000<br>
Epoch 10 | Loss: 0.1771 | Acc: 92.76% | LR: 0.001000<br>
Epoch 11 | Loss: 0.1524 | Acc: 93.41% | LR: 0.000100<br>
Epoch 12 | Loss: 0.1445 | Acc: 93.38% | LR: 0.000100<br>
Epoch 13 | Loss: 0.1399 | Acc: 93.38% | LR: 0.000100<br>
Epoch 14 | Loss: 0.1419 | Acc: 93.54% | LR: 0.000100<br>
Epoch 15 | Loss: 0.1402 | Acc: 93.55% | LR: 0.000100<br>
Epoch 16 | Loss: 0.1377 | Acc: 93.74% | LR: 0.000100<br>
Epoch 17 | Loss: 0.1355 | Acc: 93.61% | LR: 0.000100<br>
Epoch 18 | Loss: 0.1334 | Acc: 93.69% | LR: 0.000100<br>
Epoch 19 | Loss: 0.1330 | Acc: 93.59% | LR: 0.000100<br>
Epoch 20 | Loss: 0.1313 | Acc: 93.71% | LR: 0.000100<br>
训练全部完成！ <br>

这个算法的上限94.8%到95%左右，有感兴趣的可以试试